# Drop Redundant Metric Layers & Save v4 Parquet

After the deep dive in notebook 04, we keep **only** the columns needed for modelling:

| Keep | Description |
|------|-------------|
| **Structural / ID** | `wy_player_id`, height, weight, age, transfer value/fee, minutes, position, season, team, comp metadata |
| **Twelve Quality Scores** (20 from + 20 to) | Composite metrics computed by Twelve Football |
| **Team stats** (from + to) | Team-level performance metrics |

**Dropped:**
- Other quality / ratio metrics (Wyscout-derived: Aerials won %, Shot conversion %, etc.)
- Per-90 counting stats
- Position-adjusted z-scores

These layers are redundant because the Twelve Quality Scores are already computed from z-scores, which in turn come from the per-90 stats. Keeping all layers would introduce multicollinearity.

**Input:** `within_league_transfers_v3.parquet`  
**Output:** `within_league_transfers_v4.parquet`

In [ ]:
import pandas as pd
import os

INPUT_PATH  = "../../../thesis_data/processed_data/thesis_model_dataset/within_league_transfers_v3.parquet"
OUTPUT_PATH = "../../../thesis_data/processed_data/thesis_model_dataset/within_league_transfers_v4.parquet"

df = pd.read_parquet(INPUT_PATH)
print(f"Input: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
# --- Define the 20 Twelve Quality Score names ---
TWELVE_QS_NAMES = [
    "Active defence", "Aerial threat", "Box threat", "Chance prevention",
    "Composure", "Defensive heading", "Dribbling", "Effectiveness",
    "Finishing", "Hold-up play", "Intelligent defence", "Involvement",
    "Passing quality", "Poaching", "Pressing", "Progression",
    "Providing teammates", "Run quality", "Territorial dominance", "Winning duels",
]

from_twelve_qs = [f"from_{n}" for n in TWELVE_QS_NAMES]
to_twelve_qs   = [f"to_{n}" for n in TWELVE_QS_NAMES]

# --- Structural columns ---
STRUCTURAL = [
    "wy_player_id", "wyscout_height", "wyscout_weight", "player_season_age",
    "tm_transfer_value", "tm_transfer_fee",
    "from_Minutes", "from_competition", "from_position", "from_season", "from_team_id",
    "to_Minutes", "to_competition", "to_position", "to_season", "to_team_id",
    "from_comp_division", "from_comp_season_id", "from_comp_season_name",
    "to_comp_division", "to_comp_season_id", "to_comp_season_name",
]

# --- Team stats ---
from_team_stats = sorted([c for c in df.columns if c.startswith("from_team_stats_")])
to_team_stats   = sorted([c for c in df.columns if c.startswith("to_team_stats_")])

# --- Build keep list ---
KEEP = STRUCTURAL + from_twelve_qs + to_twelve_qs + from_team_stats + to_team_stats

# Verify all exist
missing = [c for c in KEEP if c not in df.columns]
if missing:
    print(f"WARNING — missing columns: {missing}")
else:
    print(f"All {len(KEEP)} columns to keep found in dataframe.")

In [ ]:
# --- What gets dropped ---
dropped = sorted(set(df.columns) - set(KEEP))

print(f"Columns KEPT:    {len(KEEP)}")
print(f"Columns DROPPED: {len(dropped)}")
print(f"\nDropped columns by type:")

drop_zscores = [c for c in dropped if "z_score" in c]
drop_per90   = [c for c in dropped if c.endswith("_per_90")]
drop_other   = [c for c in dropped if c not in drop_zscores and c not in drop_per90]

print(f"  Z-scores:              {len(drop_zscores)}")
print(f"  Per-90 stats:          {len(drop_per90)}")
print(f"  Other quality/ratio:   {len(drop_other)}")

if drop_other:
    print(f"\nOther quality/ratio columns dropped:")
    for c in drop_other:
        print(f"    {c}")

In [ ]:
# --- Select & save ---
df_v4 = df[KEEP].copy()

print(f"Output: {df_v4.shape[0]:,} rows x {df_v4.shape[1]} columns")
print(f"\nColumn groups:")
print(f"  Structural:            {len(STRUCTURAL)}")
print(f"  Twelve QS (from):      {len(from_twelve_qs)}")
print(f"  Twelve QS (to):        {len(to_twelve_qs)}")
print(f"  Team stats (from):     {len(from_team_stats)}")
print(f"  Team stats (to):       {len(to_team_stats)}")
print(f"  TOTAL:                 {df_v4.shape[1]}")

In [ ]:
# Quick sanity — null summary
null_pct = (100 * df_v4.isnull().sum() / len(df_v4)).round(1)
has_nulls = null_pct[null_pct > 0]

if len(has_nulls) > 0:
    print(f"Columns with nulls ({len(has_nulls)}):")
    print(has_nulls.sort_values(ascending=False).to_string())
else:
    print("No nulls in any column.")

In [ ]:
df_v4.to_parquet(OUTPUT_PATH, index=False)
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Saved: {OUTPUT_PATH}")
print(f"Size:  {size_mb:.1f} MB")
print(f"Shape: {df_v4.shape[0]:,} rows x {df_v4.shape[1]} columns")

In [ ]:
# Final column listing
print("All columns in v4:")
for i, c in enumerate(df_v4.columns, 1):
    print(f"  {i:3d}. {c}")